# Product A/B reference: one primary conversion metric

This notebook demonstrates a complete user-randomized product experiment with synthetic data. Synthetic data are intentional here: the assignment mechanism, generating probabilities, true effect, analysis plan, and decision threshold are all known. Use the separate Hillstrom notebook to see the extra interpretation work required for archived real data.

## Setup

From a bootstrapx checkout, install the notebook dependencies with `python -m pip install -e ".[pandas]" jupyter`. This notebook generates all data locally and does not access the network.

## Fix the analysis plan before outcomes exist

- **Randomization and analysis unit:** user.
- **Primary metric:** seven-day conversion per assigned user.
- **Estimand:** treatment minus control conversion rate.
- **Uncertainty:** two-sided 95% BCa interval with 4,999 resamples.
- **Minimum practical effect:** +0.5 percentage points.
- **Metric rule:** the lower interval endpoint must exceed +0.5 percentage points.

The constants below encode this plan before the generator creates outcomes. The metric rule is only one input to a product decision; costs, guardrails, implementation risk, and external validity remain separate.

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd

from bootstrapx import bootstrap_two_sample

N_USERS = 100_000
PRIMARY_METRIC = "converted_7d"
CONTROL_RATE = 0.10
TREATMENT_RATE = 0.112
TRUE_EFFECT = TREATMENT_RATE - CONTROL_RATE
MIN_PRACTICAL_EFFECT = 0.005
N_RESAMPLES = 4_999
EXPERIMENT_SEED = 0
BOOTSTRAP_SEED = 42

## Randomize users and generate the observed metric

Each user is assigned exactly once. Conditional on assignment, the outcome is a Bernoulli draw with a 10.0% control probability or an 11.2% treatment probability. The true superpopulation effect is therefore +1.2 percentage points; unlike a real experiment, we know it because we wrote the generator.

In [2]:
rng = np.random.default_rng(EXPERIMENT_SEED)
assignment = rng.permutation(
    np.repeat(["control", "treatment"], N_USERS // 2)
)
outcome_probability = np.where(
    assignment == "control", CONTROL_RATE, TREATMENT_RATE
)
experiment = pd.DataFrame(
    {
        "user_id": np.arange(N_USERS),
        "variant": assignment,
        PRIMARY_METRIC: rng.binomial(1, outcome_probability),
    }
)
experiment.head()

,user_id,variant,converted_7d
0,0,control,0
1,1,control,0
2,2,treatment,0
3,3,treatment,0
4,4,control,0


## Validate the analysis table

These checks protect the assumptions used by the example: one row per randomized user, two expected variants, and one finite binary primary outcome. A production pipeline should additionally validate sample ratios, exposure delivery, bots, missing events, and logging changes using platform-specific information.

In [3]:
assert len(experiment) == N_USERS
assert experiment["user_id"].is_unique
assert set(experiment["variant"]) == {"control", "treatment"}
assert experiment[PRIMARY_METRIC].isin([0, 1]).all()
assert not experiment[["variant", PRIMARY_METRIC]].isna().any().any()

arm_summary = experiment.groupby("variant", observed=True).agg(
    users=("user_id", "size"),
    conversions=(PRIMARY_METRIC, "sum"),
    conversion_rate=(PRIMARY_METRIC, "mean"),
)
arm_summary

,users,conversions,conversion_rate
variant,,,
control,50000,4942,0.09884
treatment,50000,5529,0.11058


## Estimate the predeclared primary effect

The two arms contain different users, so the default independent resampling design is appropriate. Absolute percentage-point difference is the primary scale; no outcome or alternative metric is selected after seeing the generated result.

In [4]:
control = experiment.loc[
    experiment["variant"] == "control", PRIMARY_METRIC
].to_numpy(dtype=float)
treatment = experiment.loc[
    experiment["variant"] == "treatment", PRIMARY_METRIC
].to_numpy(dtype=float)

result = bootstrap_two_sample(
    control,
    treatment,
    np.mean,
    effect="difference",
    method="bca",
    confidence_level=0.95,
    n_resamples=N_RESAMPLES,
    batch_size=100,
    random_state=BOOTSTRAP_SEED,
)

In [5]:
interval = result.confidence_interval
metric_rule_passes = interval.low > MIN_PRACTICAL_EFFECT
summary = pd.Series(
    {
        "control_rate": result.control_estimate,
        "treatment_rate": result.treatment_estimate,
        "estimated_effect": result.estimate,
        "ci_low": interval.low,
        "ci_high": interval.high,
        "known_generating_effect": TRUE_EFFECT,
        "minimum_practical_effect": MIN_PRACTICAL_EFFECT,
        "metric_rule_passes": metric_rule_passes,
    },
    name="value",
)
summary

control_rate                 0.09884
treatment_rate               0.11058
estimated_effect             0.01174
ci_low                      0.007893
ci_high                      0.01546
known_generating_effect        0.012
minimum_practical_effect       0.005
metric_rule_passes              True
Name: value, dtype: object

## Interpret this run

The deterministic run estimates control conversion at **9.884%**, treatment conversion at **11.058%**, and the effect at **+1.174 percentage points**. Its 95% BCa interval is **[+0.789, +1.546] percentage points**, which contains the known +1.200 pp generating effect.

Because the lower endpoint exceeds the predeclared +0.5 pp minimum practical effect, the primary-metric rule passes. This is not an automatic launch decision: a real team would combine it with guardrails, implementation risk, cost, and evidence that assignment and measurement operated correctly.

## Run a small known-truth coverage smoke check

A single interval containing the truth does not establish coverage. The loop below runs 100 smaller independent experiments to verify the complete known-truth workflow. It uses only 499 resamples per interval to stay fast, so it is **not a release-grade coverage study**. The [versioned benchmark suite](../docs/benchmarks.md) provides the release evidence.

In [6]:
COVERAGE_TRIALS = 100
SMOKE_USERS_PER_ARM = 400
SMOKE_RESAMPLES = 499
coverage_rng = np.random.default_rng(2026)
covered = 0

for trial in range(COVERAGE_TRIALS):
    simulated_control = coverage_rng.binomial(
        1, CONTROL_RATE, SMOKE_USERS_PER_ARM
    )
    simulated_treatment = coverage_rng.binomial(
        1, TREATMENT_RATE, SMOKE_USERS_PER_ARM
    )
    simulated_result = bootstrap_two_sample(
        simulated_control,
        simulated_treatment,
        np.mean,
        effect="difference",
        method="bca",
        n_resamples=SMOKE_RESAMPLES,
        random_state=trial,
    )
    simulated_interval = simulated_result.confidence_interval
    covered += simulated_interval.low <= TRUE_EFFECT <= simulated_interval.high

coverage_rate = covered / COVERAGE_TRIALS
coverage_mcse = np.sqrt(
    coverage_rate * (1 - coverage_rate) / COVERAGE_TRIALS
)
pd.Series(
    {
        "covered_trials": covered,
        "total_trials": COVERAGE_TRIALS,
        "coverage_rate": coverage_rate,
        "monte_carlo_standard_error": coverage_mcse,
    },
    name="smoke_check",
)

covered_trials                 94.000000
total_trials                  100.000000
coverage_rate                   0.940000
monte_carlo_standard_error      0.023749
Name: smoke_check, dtype: float64

## What this example establishes

This notebook makes the complete path inspectable: the user is the randomization and analysis unit, the primary metric and decision threshold precede outcomes, the effect is estimated directly, and known truth is separated from repeated-sampling coverage. It demonstrates the API and a defensible workflow; it does not claim that a synthetic experiment captures every production logging, interference, novelty, or business-risk problem.